In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW = Path("../data/raw")
CANCERS = ["HNSC", "LUAD", "BRCA", "COAD"]

clinical = {c: pd.read_csv(RAW / f"{c}_clinical.csv", low_memory=False) for c in CANCERS}
mutations = {c: pd.read_csv(RAW / f"{c}_mutations.csv", low_memory=False) for c in CANCERS}

for c in CANCERS:
    print(f"{c}: {len(clinical[c]):>5d} patients, {len(mutations[c]):>5d} mutations, "
          f"{clinical[c].shape[1]} clinical cols")

HNSC:   523 patients,  1437 mutations, 42 clinical cols
LUAD:   566 patients,  1362 mutations, 41 clinical cols
BRCA:  1084 patients,  1716 mutations, 42 clinical cols
COAD:   594 patients,  2483 mutations, 43 clinical cols


In [2]:
# Look at HNSC's clinical columns
print(clinical["HNSC"].columns.tolist())

['patientId', 'AGE', 'AJCC_PATHOLOGIC_TUMOR_STAGE', 'AJCC_STAGING_EDITION', 'BUFFA_HYPOXIA_SCORE', 'CANCER_TYPE_ACRONYM', 'DAYS_LAST_FOLLOWUP', 'DAYS_TO_BIRTH', 'DAYS_TO_INITIAL_PATHOLOGIC_DIAGNOSIS', 'DFS_MONTHS', 'DFS_STATUS', 'DSS_MONTHS', 'DSS_STATUS', 'ETHNICITY', 'FORM_COMPLETION_DATE', 'GENETIC_ANCESTRY_LABEL', 'HISTORY_NEOADJUVANT_TRTYN', 'ICD_10', 'ICD_O_3_HISTOLOGY', 'ICD_O_3_SITE', 'INFORMED_CONSENT_VERIFIED', 'IN_PANCANPATHWAYS_FREEZE', 'NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT', 'OS_MONTHS', 'OS_STATUS', 'OTHER_PATIENT_ID', 'PATH_M_STAGE', 'PATH_N_STAGE', 'PATH_T_STAGE', 'PERSON_NEOPLASM_CANCER_STATUS', 'PFS_MONTHS', 'PFS_STATUS', 'PRIMARY_LYMPH_NODE_PRESENTATION_ASSESSMENT', 'PRIOR_DX', 'RACE', 'RADIATION_THERAPY', 'RAGNUM_HYPOXIA_SCORE', 'SAMPLE_COUNT', 'SEX', 'SUBTYPE', 'WINTER_HYPOXIA_SCORE', 'cancer_type']


In [3]:
key_cols = ["patientId", "AGE", "SEX", "OS_MONTHS", "OS_STATUS",
            "AJCC_PATHOLOGIC_TUMOR_STAGE", "RACE"]
existing = [c for c in key_cols if c in clinical["HNSC"].columns]
clinical["HNSC"][existing].head(10)

,patientId,AGE,SEX,OS_MONTHS,OS_STATUS,AJCC_PATHOLOGIC_TUMOR_STAGE,RACE
0,TCGA-4P-AA8J,66.0,Male,3.353388,0:LIVING,STAGE IVA,Black or African American
1,TCGA-BA-4074,69.0,Male,15.188875,1:DECEASED,STAGE IVA,White
2,TCGA-BA-4076,39.0,Male,13.643686,1:DECEASED,NaN,White
3,TCGA-BA-4078,83.0,Male,9.073873,1:DECEASED,NaN,White
4,TCGA-BA-5149,47.0,Male,26.498340,1:DECEASED,STAGE IVA,White
5,TCGA-BA-5151,72.0,Male,23.736726,0:LIVING,STAGE IVA,White
6,TCGA-BA-5152,56.0,Male,42.344741,0:LIVING,STAGE IVA,White
7,TCGA-BA-5153,51.0,Male,57.928132,1:DECEASED,NaN,White
8,TCGA-BA-5555,54.0,Male,17.095703,0:LIVING,STAGE IVA,Black or African American
9,TCGA-BA-5556,58.0,Female,23.835355,0:LIVING,STAGE II,White


In [4]:
def clean_clinical(df: pd.DataFrame, cancer: str) -> pd.DataFrame:
    """Standardise clinical data: extract key columns, parse survival, clean stage."""
    # Pick only the columns we care about, but only if they exist in this cancer's data
    wanted = ["patientId", "AGE", "SEX", "RACE", "ETHNICITY",
              "OS_MONTHS", "OS_STATUS", "AJCC_PATHOLOGIC_TUMOR_STAGE",
              "TUMOR_STATUS"]
    keep = [c for c in wanted if c in df.columns]
    out = df[keep].copy()

    # Parse survival status from "1:DECEASED" / "0:LIVING" into a clean 0/1 event flag
    if "OS_STATUS" in out.columns:
        out["event"] = out["OS_STATUS"].astype(str).str.contains("DECEASED").astype(int)
    if "OS_MONTHS" in out.columns:
        out["OS_MONTHS"] = pd.to_numeric(out["OS_MONTHS"], errors="coerce")

    # Convert age to numeric
    if "AGE" in out.columns:
        out["AGE"] = pd.to_numeric(out["AGE"], errors="coerce")

    # Standardise stage: extract roman numeral I/II/III/IV from messy strings
    if "AJCC_PATHOLOGIC_TUMOR_STAGE" in out.columns:
        s = out["AJCC_PATHOLOGIC_TUMOR_STAGE"].astype(str).str.upper()
        out["STAGE_SIMPLE"] = (s.str.extract(r"(IV|III|II|I)\b")[0]
                                .fillna("Unknown"))

    out["cancer_type"] = cancer
    return out


clinical_clean = pd.concat([clean_clinical(clinical[c], c) for c in CANCERS],
                            ignore_index=True)
print(f"Combined clinical: {len(clinical_clean):,} patients")
print(f"\nStage distribution:")
print(clinical_clean["STAGE_SIMPLE"].value_counts())
print(f"\nEvent (death) rate by cancer:")
print(clinical_clean.groupby("cancer_type")["event"].agg(["count", "sum", "mean"]).round(3))

Combined clinical: 2,767 patients

Stage distribution:
STAGE_SIMPLE
Unknown    2216
I           224
II          119
III         104
IV          104
Name: count, dtype: int64

Event (death) rate by cancer:
             count  sum   mean
cancer_type                   
BRCA          1084  151  0.139
COAD           594  120  0.202
HNSC           523  219  0.419
LUAD           566  186  0.329


In [6]:
# Debug: see what mutation columns actually exist
print("HNSC mutations columns:")
print(mutations["HNSC"].columns.tolist())
print()
print("First mutation record:")
print(mutations["HNSC"].iloc[0].to_dict())

HNSC mutations columns:
['uniqueSampleKey', 'uniquePatientKey', 'molecularProfileId', 'sampleId', 'patientId', 'entrezGeneId', 'gene', 'studyId', 'center', 'mutationStatus', 'validationStatus', 'tumorAltCount', 'tumorRefCount', 'normalAltCount', 'normalRefCount', 'startPosition', 'endPosition', 'referenceAllele', 'proteinChange', 'mutationType', 'ncbiBuild', 'variantType', 'keyword', 'chr', 'variantAllele', 'refseqMrnaId', 'proteinPosStart', 'proteinPosEnd', 'cancer_type']

First mutation record:
{'uniqueSampleKey': 'VENHQS00UC1BQThKLTAxOmhuc2NfdGNnYV9wYW5fY2FuX2F0bGFzXzIwMTg', 'uniquePatientKey': 'VENHQS00UC1BQThKOmhuc2NfdGNnYV9wYW5fY2FuX2F0bGFzXzIwMTg', 'molecularProfileId': 'hnsc_tcga_pan_can_atlas_2018_mutations', 'sampleId': 'TCGA-4P-AA8J-01', 'patientId': 'TCGA-4P-AA8J', 'entrezGeneId': 2195, 'gene': "{'entrezGeneId': 2195, 'hugoGeneSymbol': 'FAT1', 'type': 'protein-coding'}", 'studyId': 'hnsc_tcga_pan_can_atlas_2018', 'center': '.', 'mutationStatus': '.', 'validationStatus': '.'

In [7]:
import ast

def patient_id_from_sample(sample_id: str) -> str:
    """TCGA sample IDs like 'TCGA-CN-4733-01' -> patient ID 'TCGA-CN-4733'."""
    parts = str(sample_id).split("-")
    return "-".join(parts[:3]) if len(parts) >= 3 else str(sample_id)


def _extract_gene_symbol(row) -> str:
    """Get gene symbol from either flat column or nested gene dict."""
    if "hugoGeneSymbol" in row.index and pd.notna(row.get("hugoGeneSymbol")):
        return row["hugoGeneSymbol"]
    g = row.get("gene")
    if pd.isna(g):
        return "UNKNOWN"
    if isinstance(g, dict):
        return g.get("hugoGeneSymbol", "UNKNOWN")
    # gene is a string like "{'hugoGeneSymbol': 'TP53', ...}"
    try:
        return ast.literal_eval(str(g)).get("hugoGeneSymbol", "UNKNOWN")
    except Exception:
        return "UNKNOWN"


def aggregate_mutations(mut_df: pd.DataFrame) -> pd.DataFrame:
    """One row per patient with mut_<GENE> indicators + total mutation count."""
    df = mut_df.copy()
    if df.empty:
        return pd.DataFrame(columns=["patientId"])

    # 1. Get patient ID
    if "patientId" in df.columns:
        pid_col = "patientId"
    elif "sampleId" in df.columns:
        df["patientId"] = df["sampleId"].apply(patient_id_from_sample)
        pid_col = "patientId"
    else:
        raise ValueError(
            f"No patient/sample ID column. Available: {df.columns.tolist()[:15]}"
        )

    # 2. Get gene symbol (handles flat or nested 'gene' field)
    df["gene_symbol"] = df.apply(_extract_gene_symbol, axis=1)

    # 3. Pivot: one row per patient, one column per gene
    pivot = (df.assign(_one=1)
               .pivot_table(index=pid_col,
                            columns="gene_symbol",
                            values="_one",
                            aggfunc="max",
                            fill_value=0)
               .astype(int))
    pivot.columns = [f"mut_{g}" for g in pivot.columns]
    pivot["total_mutations"] = pivot.sum(axis=1)
    return pivot.reset_index()


mutations_per_patient = pd.concat(
    [aggregate_mutations(mutations[c]) for c in CANCERS],
    ignore_index=True,
)
mutations_per_patient = mutations_per_patient.groupby("patientId").max().reset_index()
mutations_per_patient = mutations_per_patient.fillna(0)
print(f"Mutations table: {len(mutations_per_patient):,} patients, "
      f"{mutations_per_patient.shape[1]-1} mutation columns")
mutations_per_patient.head()

Mutations table: 2,329 patients, 31 mutation columns


,patientId,mut_APC,mut_ARID1A,mut_ATM,mut_BRAF,mut_CASP8,mut_CDH1,mut_CDKN2A,mut_EGFR,mut_EP300,...,mut_NSD1,mut_PIK3CA,mut_PIK3R1,mut_PTEN,mut_RB1,mut_SMAD4,mut_STK11,mut_TP53,total_mutations,mut_VHL
0,TCGA-05-4244,0,0,1,0,0,0,0.0,0,0,...,0,0,0,0,0,0,0,0,2,0.0
1,TCGA-05-4249,0,0,0,1,0,0,0.0,0,0,...,0,1,0,0,0,0,0,0,5,0.0
2,TCGA-05-4250,0,0,0,0,0,0,0.0,0,0,...,0,0,0,0,0,0,0,0,2,0.0
3,TCGA-05-4382,1,1,1,1,0,0,1.0,1,0,...,0,0,0,0,0,0,0,1,7,0.0
4,TCGA-05-4384,0,0,0,0,0,0,0.0,0,0,...,0,0,0,0,0,0,0,1,2,0.0


In [8]:
final = clinical_clean.merge(mutations_per_patient, on="patientId", how="left")

# Patients with no mutation records get 0 across all gene columns
mut_cols = [c for c in final.columns if c.startswith("mut_")] + ["total_mutations"]
final[mut_cols] = final[mut_cols].fillna(0).astype(int)

# Drop patients without survival info (can't use them for survival modeling)
before = len(final)
final = final.dropna(subset=["OS_MONTHS", "event"])
final = final[final["OS_MONTHS"] > 0]
print(f"Patients dropped due to missing/zero survival: {before - len(final)}")
print(f"Final feature matrix: {len(final):,} patients × {final.shape[1]} columns")
final.head()

Patients dropped due to missing/zero survival: 105
Final feature matrix: 2,662 patients × 42 columns


,patientId,AGE,SEX,RACE,ETHNICITY,OS_MONTHS,OS_STATUS,AJCC_PATHOLOGIC_TUMOR_STAGE,event,STAGE_SIMPLE,...,mut_NSD1,mut_PIK3CA,mut_PIK3R1,mut_PTEN,mut_RB1,mut_SMAD4,mut_STK11,mut_TP53,total_mutations,mut_VHL
0,TCGA-4P-AA8J,66.0,Male,Black or African American,Not Hispanic Or Latino,3.353388,0:LIVING,STAGE IVA,0,Unknown,...,0,0,0,0,0,0,0,1,2,0
1,TCGA-BA-4074,69.0,Male,White,Not Hispanic Or Latino,15.188875,1:DECEASED,STAGE IVA,1,Unknown,...,0,0,0,1,0,0,0,1,3,0
2,TCGA-BA-4076,39.0,Male,White,Not Hispanic Or Latino,13.643686,1:DECEASED,NaN,1,Unknown,...,0,0,0,0,0,0,0,1,5,0
3,TCGA-BA-4078,83.0,Male,White,Not Hispanic Or Latino,9.073873,1:DECEASED,NaN,1,Unknown,...,0,0,0,0,0,0,0,1,3,0
4,TCGA-BA-5149,47.0,Male,White,Not Hispanic Or Latino,26.498340,1:DECEASED,STAGE IVA,1,Unknown,...,0,1,0,0,0,0,0,0,4,0


In [9]:
print("=== Survival distribution by cancer ===")
print(final.groupby("cancer_type").agg(
    n=("patientId", "count"),
    median_os_months=("OS_MONTHS", "median"),
    event_rate=("event", "mean"),
).round(3))

print("\n=== Top 10 most-mutated genes ===")
mut_counts = final[[c for c in final.columns if c.startswith("mut_")]].sum().sort_values(ascending=False)
print(mut_counts.head(10))

print("\n=== TP53 mutation prevalence by cancer ===")
print(final.groupby("cancer_type")["mut_TP53"].mean().round(3))

=== Survival distribution by cancer ===
                n  median_os_months  event_rate
cancer_type                                    
BRCA         1071            27.912       0.141
COAD          568            22.044       0.204
HNSC          522            21.189       0.420
LUAD          501            21.567       0.361

=== Top 10 most-mutated genes ===
mut_TP53      1263
mut_PIK3CA     598
mut_APC        431
mut_KRAS       372
mut_KMT2C      249
mut_FAT1       243
mut_KMT2D      205
mut_GATA3      159
mut_CDH1       158
mut_ATM        147
dtype: int64

=== TP53 mutation prevalence by cancer ===
cancer_type
BRCA    0.320
COAD    0.533
HNSC    0.684
LUAD    0.519
Name: mut_TP53, dtype: float64


In [10]:
out = Path("../data/processed")
out.mkdir(exist_ok=True)
final.to_csv(out / "tcga_features.csv", index=False)
print(f"Saved {len(final):,} patients to {out / 'tcga_features.csv'}")

Saved 2,662 patients to ../data/processed/tcga_features.csv
